# 07 · Case Study: Retail Sales Analytics

> **Module 3 · Case Studies · Lecture 3.1**

In this notebook you will:

- Generate deterministic POS sales data so the notebook is fully self-contained
- Deduplicate transactions with a deterministic `row_number` window
- Enrich sales with broadcast dimension joins
- Aggregate to a daily sales summary and assert data-quality thresholds
- Write the result to local Parquet, with an optional BigQuery load

---


## Setup / prerequisites

- **Python 3.9–3.11** and **Java 11 or 17** on your machine.
- **PySpark 3.5.x**: `pip install "pyspark==3.5.1" pandas pyarrow numpy`
- This notebook is **self-contained and offline**: every code cell runs top-to-bottom with
  no GCP account. All data operations use local paths under `/tmp/`.
- GCS / BigQuery / Pub/Sub / JDBC code appears in **clearly-marked optional cells** that
  you uncomment after authenticating with `gcloud auth application-default login`.
- On Dataproc, **do not `pip install pyspark`** — use the Spark runtime already on the
  image (Spark 3.5 on Dataproc 2.2).


## 1 · The business problem

A retail chain records point-of-sale (POS) transactions. Each transaction may be
re-transmitted by the store network, so the same `transaction_id` can appear more than
once. The analytics team needs a daily sales summary joined to customer and product
dimensions, with hard quality gates so that bad data never reaches the dashboard.

The pipeline:

```
bronze (raw POS)  ->  silver (deduped, enriched)  ->  gold (daily summary)
```

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pyspark

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("notebook-01")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("PySpark :", pyspark.__version__)
print("Spark   :", spark.version)
print("Master  :", spark.sparkContext.master)


## 2 · Generate deterministic sample data

`F.rand(seed)` makes the data reproducible: the same seed gives the same rows on every
run, on every machine. That is what makes the quality assertions below meaningful.

In [ ]:
import os
import shutil

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, DateType, TimestampType,
)

DATA_DIR = "/tmp/dataproc_notebook_07_retail"
if os.path.exists(DATA_DIR):
    shutil.rmtree(DATA_DIR)
os.makedirs(DATA_DIR, exist_ok=True)

SEED = 42
N_TRANSACTIONS = 2000

# Deterministic POS transactions. F.rand(seed) is reproducible across runs.
pos_raw = (
    spark.range(N_TRANSACTIONS)
    .withColumn("transaction_id", F.concat(F.lit("tx-"), F.col("id").cast("string")))
    .withColumn("customer_id", F.concat(F.lit("cust-"), (F.col("id") % 50).cast("string")))
    .withColumn("product_id", F.concat(F.lit("prod-"), (F.col("id") % 12).cast("string")))
    .withColumn("store_id", F.concat(F.lit("store-"), (F.col("id") % 8).cast("string")))
    .withColumn("quantity", (F.rand(SEED) * 5 + 1).cast("int"))
    .withColumn("unit_price", F.round(F.rand(SEED + 1) * 200 + 5, 2))
    .withColumn("discount_pct", F.round(F.rand(SEED + 2) * 0.2, 4))
    .withColumn("sale_date", F.date_add(F.lit("2026-09-01").cast("date"),
                                        (F.col("id") % 30).cast("int")))
    .withColumn("sale_ts", F.to_timestamp(F.concat(
        F.col("sale_date").cast("string"), F.lit(" 10:00:00"))))
)

# Injected duplicates: the store network re-sends ~5% of transactions.
duplicated = pos_raw.sample(withReplacement=False, fraction=0.05, seed=SEED)
pos_with_duplicates = (
    pos_raw.unionByName(duplicated)
    .withColumn("ingest_ts", F.current_timestamp())
)

print("raw rows (with duplicates):", pos_with_duplicates.count())
print("distinct transaction_id   :", pos_with_duplicates.select("transaction_id").distinct().count())
pos_with_duplicates.show(5, truncate=False)

In [ ]:
# Dimensions: customers and products.
customers = (
    spark.range(50)
    .withColumn("customer_id", F.concat(F.lit("cust-"), F.col("id").cast("string")))
    .withColumn("customer_name", F.concat(F.lit("Customer "), F.col("id").cast("string")))
    .withColumn("loyalty_tier", F.when(F.col("id") % 10 == 0, F.lit("gold"))
                                    .when(F.col("id") % 3 == 0, F.lit("silver"))
                                    .otherwise(F.lit("bronze")))
    .withColumn("signup_date", F.date_add(F.lit("2024-01-01").cast("date"),
                                          (F.col("id") % 500).cast("int")))
    .drop("id")   # drop the surrogate key, or the join produces two `id` columns
)

products = (
    spark.range(12)
    .withColumn("product_id", F.concat(F.lit("prod-"), F.col("id").cast("string")))
    .withColumn("product_name", F.concat(F.lit("Product "), F.col("id").cast("string")))
    .withColumn("category", F.concat(F.lit("Category "), (F.col("id") % 4).cast("string")))
    .withColumn("unit_cost", F.round(F.rand(SEED + 3) * 120 + 2, 2))
    .drop("id")   # drop the surrogate key, or the join produces two `id` columns
)

print("customers:", customers.count(), " products:", products.count())
customers.show(3, truncate=False)
products.show(3, truncate=False)

## 3 · Bronze -> Silver: dedupe and enrich

**Dedupe.** `dropDuplicates(subset=["transaction_id"])` keeps an *arbitrary* row. For an
auditable pipeline you want the **latest** ingested copy of each transaction, so use
`row_number` with a deterministic tiebreaker.

**Enrich.** The dimensions are tiny (50 and 12 rows), so `F.broadcast` eliminates the
shuffle entirely.

In [ ]:
from pyspark.sql import Window

# Deterministic "keep the latest ingest, break ties by transaction_id".
dedupe_window = Window.partitionBy("transaction_id").orderBy(
    F.col("ingest_ts").desc(), F.col("transaction_id").desc()
)

silver = (
    pos_with_duplicates
    .withColumn("rn", F.row_number().over(dedupe_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
    # Derived measures.
    .withColumn("gross_amount", F.round(F.col("quantity") * F.col("unit_price"), 2))
    .withColumn("discount_amount", F.round(F.col("gross_amount") * F.col("discount_pct"), 2))
    .withColumn("net_amount", F.round(F.col("gross_amount") - F.col("discount_amount"), 2))
)

print("silver rows:", silver.count())
silver.show(5, truncate=False)

In [ ]:
# Broadcast joins: the dimensions are tiny, so no shuffle is needed.
silver_enriched = (
    silver
    .join(F.broadcast(customers), "customer_id", "inner")
    .join(F.broadcast(products), "product_id", "inner")
    .withColumn("gross_margin", F.round(F.col("net_amount") - F.col("quantity") * F.col("unit_cost"), 2))
)

print("enriched rows:", silver_enriched.count())
silver_enriched.select(
    "transaction_id", "customer_name", "product_name", "category",
    "quantity", "net_amount", "gross_margin",
).show(5, truncate=False)

silver_enriched.explain(mode="formatted")

## 4 · Gold: the daily sales summary

The gold layer is an aggregate that a dashboard can query directly: one row per
`sale_date × category × loyalty_tier`.

In [ ]:
daily_sales = (
    silver_enriched
    .groupBy(
        F.col("sale_date").alias("sale_date"),
        F.col("category").alias("category"),
        F.col("loyalty_tier").alias("loyalty_tier"),
    )
    .agg(
        F.count("*").alias("transaction_count"),
        F.countDistinct("transaction_id").alias("distinct_transactions"),
        F.countDistinct("customer_id").alias("distinct_customers"),
        F.sum("quantity").alias("units_sold"),
        F.round(F.sum("net_amount"), 2).alias("net_revenue"),
        F.round(F.sum("discount_amount"), 2).alias("total_discount"),
        F.round(F.sum("gross_margin"), 2).alias("gross_margin"),
        F.round(F.avg("net_amount"), 2).alias("avg_order_value"),
    )
    .orderBy("sale_date", "category", "loyalty_tier")
)

print("gold rows:", daily_sales.count())
daily_sales.show(20, truncate=False)

## 5 · Quality assertions

A pipeline without assertions is a pipeline that will quietly publish wrong numbers.
These assertions raise loudly when the data violates the contract.

**Caution:** `assert` statements are removed when Python runs with `-O`. Use explicit `raise` in
production code.

In [ ]:
def report_check(name, ok, detail):
    print(f"  [{'PASS' if ok else 'FAIL'}] {name:32s} {detail}")
    if not ok:
        raise RuntimeError(f"Quality check failed: {name} ({detail})")
    return True


def assert_silver_quality(df, tolerance=0.01):
    """Row-level checks on the silver (transaction-grain) layer."""
    row_count = df.count()
    report_check("row_count > 0", row_count > 0, f"{row_count} rows")

    null_ids = df.filter(F.col("transaction_id").isNull()).count()
    report_check("no null transaction_id", null_ids == 0, f"{null_ids} null ids")

    negative = df.filter(F.col("net_amount") < 0).count()
    report_check("no negative net_amount", negative == 0, f"{negative} negatives")

    # net_amount must equal gross - discount, within tolerance.
    mismatched = df.filter(
        F.abs((F.col("gross_amount") - F.col("discount_amount")) - F.col("net_amount")) > tolerance
    ).count()
    report_check("amounts reconcile", mismatched == 0, f"{mismatched} mismatches")

    # The silver layer must be de-duplicated: one row per transaction_id.
    distinct_ids = df.select("transaction_id").distinct().count()
    report_check("one row per transaction_id", distinct_ids == row_count,
                 f"{distinct_ids} distinct / {row_count} rows")
    return True


def assert_gold_quality(df, min_rows=1):
    """Aggregate-level checks on the gold (daily-grain) layer."""
    row_count = df.count()
    report_check("gold row_count > 0", row_count > min_rows, f"{row_count} rows")

    negative = df.filter(F.col("net_revenue") < 0).count()
    report_check("no negative net_revenue", negative == 0, f"{negative} negatives")

    null_dates = df.filter(F.col("sale_date").isNull()).count()
    report_check("no null sale_date", null_dates == 0, f"{null_dates} null dates")

    # A daily grain means one row per (sale_date, category, loyalty_tier).
    groups = df.select("sale_date", "category", "loyalty_tier").distinct().count()
    report_check("grain is unique", groups == row_count, f"{groups} groups / {row_count} rows")
    return True


print("== silver quality gate ==")
assert_silver_quality(silver_enriched)

print("\n== gold quality gate ==")
assert_gold_quality(daily_sales)

## 6 · Write to Parquet on local disk

On Dataproc the same call writes to `gs://your-bucket/...`.

In [ ]:
silver_path = f"{DATA_DIR}/silver/pos_sales"
gold_path = f"{DATA_DIR}/gold/daily_sales"

(
    silver_enriched.write
    .mode("overwrite")
    .option("compression", "snappy")
    .partitionBy("sale_date")
    .parquet(silver_path)
)
daily_sales.write.mode("overwrite").option("compression", "snappy").parquet(gold_path)

print("silver files:", sum(len([f for f in fs if f.endswith(".parquet")])
                            for _, _, fs in os.walk(silver_path)))
print("gold files  :", len([f for f in os.listdir(gold_path) if f.endswith(".parquet")]))

# Read back and verify the round trip.
gold_roundtrip = spark.read.parquet(gold_path)
print("\nrows read back:", gold_roundtrip.count())
gold_roundtrip.orderBy("sale_date").show(5, truncate=False)

### Optional: load to BigQuery

Uncomment after authenticating with `gcloud auth application-default login`.

In [ ]:
# ------------------------------------------------------------------
# OPTIONAL — BigQuery load of the gold layer.
# ------------------------------------------------------------------
# BQ_PROJECT = "your-project-id"
# (
#     daily_sales.write
#     .format("bigquery")
#     .option("writeDisposition", "WRITE_TRUNCATE")
#     .option("temporaryGcsBucket", "your-staging-bucket")
#     .option("writeMethod", "direct")
#     .save(f"{BQ_PROJECT}.retail.daily_sales")
# )

print("BigQuery load cell is commented out — no GCP account required for this notebook.")

## 7 · Idempotent re-run

Because the data is generated from a fixed seed and every write uses `mode("overwrite")`,
running the notebook twice produces identical results.

In [ ]:
# Re-run the aggregate and confirm it is identical.
daily_sales_again = (
    silver_enriched
    .groupBy(
        F.col("sale_date").alias("sale_date"),
        F.col("category").alias("category"),
        F.col("loyalty_tier").alias("loyalty_tier"),
    )
    .agg(
        F.count("*").alias("transaction_count"),
        F.countDistinct("transaction_id").alias("distinct_transactions"),
        F.countDistinct("customer_id").alias("distinct_customers"),
        F.sum("quantity").alias("units_sold"),
        F.round(F.sum("net_amount"), 2).alias("net_revenue"),
        F.round(F.sum("discount_amount"), 2).alias("total_discount"),
        F.round(F.sum("gross_margin"), 2).alias("gross_margin"),
        F.round(F.avg("net_amount"), 2).alias("avg_order_value"),
    )
)

sig_a = daily_sales.agg(
    F.sum("net_revenue").alias("revenue"),
    F.sum("transaction_count").alias("txns"),
    F.count("*").alias("groups"),
).collect()[0]
sig_b = daily_sales_again.agg(
    F.sum("net_revenue").alias("revenue"),
    F.sum("transaction_count").alias("txns"),
    F.count("*").alias("groups"),
).collect()[0]

print("run 1:", sig_a)
print("run 2:", sig_b)
print("identical:", sig_a == sig_b)

## 🔑 Key Takeaways

- `F.rand(seed)` gives deterministic, reproducible sample data — the foundation of meaningful assertions.
- Dedupe with `row_number` over a deterministic window, not `dropDuplicates`, when the retained row must be auditable.
- `F.broadcast` a tiny dimension and the join loses its shuffle entirely.
- Derive measures explicitly (`gross`, `discount`, `net`) so the quality gate can reconcile them.
- Quality gates must **raise**, not just log — a silently degraded pipeline is worse than a failed one.
- `groupBy` + `agg` to a daily grain, then write Parquet partitioned by `sale_date`.
- With a fixed seed and `mode("overwrite")`, the pipeline is idempotent: re-running produces identical results.
- On Dataproc, the same write becomes `gs://bucket/...`; the BigQuery load is a one-line change.

---

## 📚 Next steps

- Add a `store_id` rollup to the gold layer and a quality gate on store coverage.
- Continue to `08-case-study-healthcare.ipynb` for nested JSON and windowed anomaly detection.
